# YOLOv8 Object Detection Model Training

Notebook ini digunakan untuk melatih model **YOLOv8 Object Detection** pada dataset buah kesegaran (`segar` dan `tidak_segar`) yang telah Anda unduh dari Roboflow.

### 1. Install & Verify Dependencies

In [ ]:
!pip install -r requirements.txt

### 2. Verify Hardware Accelerator (CPU vs CUDA GPU)

In [ ]:
import torch
from ultralytics import YOLO
import os
from pathlib import Path

# Check CUDA
cuda_available = torch.cuda.is_available()
device = 0 if cuda_available else "cpu"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {cuda_available}")
if cuda_available:
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
print(f"Selected YOLO training device: {device}")

### 3. Load YOLOv8 Object Detection Pre-trained Model

In [ ]:
# Load pre-trained object detection model
model_name = "yolov8n.pt"  # Nano detector model
model = YOLO(model_name)
print(f"Loaded pretrained model: {model_name}")

### 4. Execute Object Detection Training

In [ ]:
# Paths setup (Dynamic and Relative)
YOLO_VERSION_DIR = Path.cwd()
DATASET_YAML = YOLO_VERSION_DIR / "yolo_detection_dataset" / "data.yaml"
RUNS_DIR = YOLO_VERSION_DIR / "runs"

print(f"Training using configuration file: {DATASET_YAML}")
print(f"Training logs/weights saved to:     {RUNS_DIR}")

# Start Object Detection Training
results = model.train(
    data=str(DATASET_YAML),
    epochs=25,             # Epochs for training
    imgsz=640,             # Resolution size for detection
    batch=16,              # Safe batch size to prevent out-of-memory
    workers=4,
    device=device,
    project=str(RUNS_DIR), 
    name="detect_freshness",
    patience=5,            # Early stopping to prevent overfitting
    dropout=0.15,          # Regularization to prevent overfitting
    val=True               # Validate model after each epoch
)

### 5. Evaluate and Visualize Training Curves

In [ ]:
from IPython.display import Image, display

runs_dir = Path.cwd() / "runs" / "detect_freshness"

print("Training Run Directory contents:")
if runs_dir.exists():
    for path in runs_dir.iterdir():
        if path.is_file() and path.suffix in [".png", ".csv"]:
            print(f"- {path.name}")

    # Show metrics results plot
    results_plot = runs_dir / "results.png"
    if results_plot.exists():
        print("\nShowing Training Results (Accuracy & Loss Curves):")
        display(Image(filename=str(results_plot)))

    # Show Confusion Matrix
    cm_plot = runs_dir / "confusion_matrix.png"
    if cm_plot.exists():
        print("\nShowing Confusion Matrix:")
        display(Image(filename=str(cm_plot)))
else:
    print("Runs folder not found yet. Complete training first!")

### 6. Locate Trained Weights

In [ ]:
runs_dir = Path.cwd() / "runs" / "detect_freshness"
best_weights = runs_dir / "weights" / "best.pt"
last_weights = runs_dir / "weights" / "last.pt"

print(f"Best Model Weights Path: {best_weights} (Exists: {best_weights.exists()})")
print(f"Last Model Weights Path: {last_weights} (Exists: {last_weights.exists()})")